In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [ ]:

df = pd.read_csv("house_prices.csv")

df.head()

In [ ]:

print(df.head())
print(df.info())
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

In [ ]:

df = df.drop_duplicates()
df = df.dropna(subset=["price"]).copy()

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["price"])

In [ ]:

df["garage"] = (
    df["garage"].astype("string").str.strip().str.lower()
    .map({"yes": 1, "no": 0})
)

In [ ]:

print(df["garage"].value_counts(dropna=False))
print(df[["location", "condition"]].nunique())

In [ ]:

features = [
    "area", "bathrooms", "bedrooms", "floors",
    "yearbuilt", "location", "condition", "garage"
]

X = df[features].copy()
y = df["price"]

numeric_features = [
    "area", "bathrooms", "bedrooms",
    "floors", "yearbuilt", "garage"
]
categorical_features = ["location", "condition"]

In [ ]:

from sklearn.impute import SimpleImputer

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

In [ ]:

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

model.fit(X_train, y_train)

In [ ]:

y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": y_pred
})

results.head(10)

In [ ]:

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

In [ ]:

new_house = pd.DataFrame([{
    "area": 2000,
    "bathrooms": 2,
    "bedrooms": 3,
    "floors": 1,
    "yearbuilt": 2018,
    "location": "Nairobi",
    "condition": "Good",
    "garage": "Yes"
}])

new_house["garage"] = (
    new_house["garage"].str.strip().str.lower()
    .map({"yes": 1, "no": 0})
)

predicted_price = model.predict(new_house[features])[0]

print(f"Predicted House Price: {predicted_price:,.2f}")

In [ ]:

import joblib

joblib.dump(model, "house_price_model.pkl")

In [ ]:

model = joblib.load("house_price_model.pkl")

In [ ]:
GRADIENTBOOSTING
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)

In [ ]:
gb_pred = gb_model.predict(X_test)

In [ ]:
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_r2 = r2_score(y_test, gb_pred)

print(f"MAE:  {gb_mae:,.2f}")
print(f"RMSE: {gb_rmse:,.2f}")
print(f"R²:   {gb_r2:.4f}")

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "Gradient Boosting"],
    "MAE": [
        mean_absolute_error(y_test, y_pred), gb_mae
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred)), gb_rmse
    ],
    "R2 Score": [
        r2_score(y_test, y_pred), gb_r2
    ]
})

comparison.round(4)

In [ ]:
best_model_name = comparison.loc[
    comparison["RMSE"].idxmin(), "Model"
]

print("Best model based on RMSE:", best_model_name)

In [ ]:
pd.DataFrame({
    "Actual Price": y_test.iloc[:10].values,
    "Random Forest": y_pred[:10],
    "Gradient Boosting": gb_pred[:10]
})

In [ ]:
best_model = model if best_model_name == "Random Forest" else gb_model

import joblib
joblib.dump(best_model, "best_house_price_model.pkl")